# URW-Depth — Inferență și Vizualizare
Rulează celulele în ordine. **Nu este nevoie de upload manual** — toate imaginile se descarcă automat.

- **Figura 1**: URW-Depth vs URW-Depth-Weather pe imaginea din Timișoara (vreme sintetică)
- **Figura 2**: Comparație robusteță KITTI-C — Tiny-Depth vs URW-Depth vs URW-Depth-Weather

In [ ]:
# 1. Clonare repo și instalare dependențe
!git clone https://github.com/marabonea16/TinyDepth-Experiments.git TinyDepth
%cd TinyDepth
!pip install timm==1.0.25 huggingface_hub -q

import torch
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')
print('OK')

In [ ]:
# 2. Descărcare weights și imagini de test de pe HuggingFace
from huggingface_hub import hf_hub_download
import os

REPO = 'mara-bonea-16/tinydepth-experiments'

# Weights pentru cele 3 modele
MODELS = {
    'Tiny-Depth':         'Tiny-Depth-b6/models/weights_9',
    'URW-Depth':          'URW-Depth-Calibrated/models/weights_latest',
    'URW-Depth-Weather':  'URW-Depth-S2-Fix4/models/weights_latest',
}

for model_name, hf_path in MODELS.items():
    local_dir = f'models/{model_name}/weights'
    os.makedirs(local_dir, exist_ok=True)
    for fname in ['encoder.pth', 'depth.pth']:
        dest = f'{local_dir}/{fname}'
        if not os.path.exists(dest):
            print(f'Descărcând {model_name}/{fname}...')
            src = hf_hub_download(
                repo_id=REPO, repo_type='model',
                filename=f'{hf_path}/{fname}',
                local_dir=f'_tmp_{model_name}',
                local_dir_use_symlinks=False,
            )
            os.rename(src, dest)
    print(f'  {model_name} OK')

# Backbone TinyViT
backbone = 'networks/tiny_vit_5m_22k_distill_depth.pth'
if not os.path.exists(backbone):
    src = hf_hub_download(
        repo_id=REPO, repo_type='model',
        filename='Tiny-Depth-b6/models/weights_9/encoder.pth',
        local_dir='_tmp_bb', local_dir_use_symlinks=False,
    )
    os.rename(src, backbone)
    print('  Backbone OK')

# Imagini de test
SAMPLES = [
    'timisoara.jpg',
    'kitti_c_clean.png', 'kitti_c_fog5.png',
    'kitti_c_snow3.png', 'kitti_c_frost3.png',
]
os.makedirs('samples', exist_ok=True)
for s in SAMPLES:
    dest = f'samples/{s}'
    if not os.path.exists(dest):
        src = hf_hub_download(
            repo_id=REPO, repo_type='model',
            filename=f'samples/{s}',
            local_dir='_tmp_samples', local_dir_use_symlinks=False,
        )
        os.rename(src, dest)
    print(f'  {s} OK')

print('\nToate fișierele descărcate!')

In [ ]:
# 3. Import și setup
import cv2
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, '.')
import networks
from networks.configuration import get_config
from layer import disp_to_depth

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

HEIGHT, WIDTH = 192, 640

In [ ]:
# 4. Funcții de augmentare meteorologică
def apply_snow(img, severity=0.70):
    arr = np.array(img, dtype=np.float32)
    h, w = arr.shape[:2]
    gray = arr.mean(axis=2, keepdims=True)
    arr = arr * (1 - severity * 0.4) + gray * severity * 0.4
    rng = np.random.RandomState(42)
    num_flakes = int(severity * 2500)
    ys = rng.randint(0, h, num_flakes)
    xs = rng.randint(0, w, num_flakes)
    sizes = rng.randint(1, 4, num_flakes)
    alphas = rng.uniform(0.7, 1.0, num_flakes)
    for y, x, s, a in zip(ys, xs, sizes, alphas):
        for dy in range(s):
            for dx in range(s):
                arr[min(y+dy, h-1), min(x+dx, w-1)] = (
                    arr[min(y+dy, h-1), min(x+dx, w-1)] * (1 - a) + 255 * a
                )
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def apply_fog(img, severity=0.65):
    arr = np.array(img, dtype=np.float32)
    fog_color = np.array([210, 210, 210], dtype=np.float32)
    h = arr.shape[0]
    gradient = np.linspace(severity, severity * 0.3, h, dtype=np.float32)[:, None, None]
    arr = arr * (1 - gradient) + fog_color * gradient
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def apply_rain(img, severity=0.70):
    arr = np.array(img, dtype=np.float32)
    h, w = arr.shape[:2]
    rng = np.random.RandomState(42)
    for _ in range(int(severity * 2000)):
        x = rng.randint(0, w)
        y = rng.randint(0, h - 25)
        length = rng.randint(15, 40)
        alpha = rng.uniform(0.5, 0.85)
        for k in range(length):
            yi = min(y + k, h - 1)
            xi = min(x + k // 3, w - 1)
            arr[yi, xi] = arr[yi, xi] * (1 - alpha) + 200 * alpha
    arr = arr * (1 - severity * 0.2) + 80 * severity * 0.2
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

print('Funcții augmentare definite.')

In [ ]:
# 5. Încărcare modele
def load_model(weights_dir, use_suppression=False):
    class _Opt:
        img_height = HEIGHT; img_width = WIDTH
        encoder = 'tiny_vit_5m_22k_distill'; scales = [0]
    config = get_config(_Opt())
    enc = networks.build_model(config, img_width=WIDTH, img_height=HEIGHT)
    enc_dict = torch.load(f'{weights_dir}/encoder.pth', map_location=device)
    enc.load_state_dict({k: v for k, v in enc_dict.items() if k in enc.state_dict()})
    enc.to(device).eval()
    dec = networks.FusionDecoder(
        [64, 64, 128, 160, 320],
        use_feature_suppression=use_suppression,
        gate_depth_input=False,
    )
    dec.load_state_dict(
        torch.load(f'{weights_dir}/depth.pth', map_location=device), strict=False
    )
    dec.to(device).eval()
    return enc, dec

print('Încărcare Tiny-Depth (baseline)...')
enc_tiny, dec_tiny = load_model('models/Tiny-Depth/weights', use_suppression=False)

print('Încărcare URW-Depth...')
enc_urw, dec_urw = load_model('models/URW-Depth/weights', use_suppression=False)

print('Încărcare URW-Depth-Weather...')
enc_w, dec_w = load_model('models/URW-Depth-Weather/weights', use_suppression=True)

print('Toate modelele încărcate!')

In [ ]:
# 6. Pregătire imagini — automat, fără upload manual

# --- Timișoara (descărcată în celula 2) ---
img_orig = Image.open('samples/timisoara.jpg').convert('RGB')

images_timisoara = {
    'Original': img_orig,
    'Ninsoare': apply_snow(img_orig),
    'Ceată':    apply_fog(img_orig),
    'Ploaie':   apply_rain(img_orig),
}
print('Timișoara pregătită:', list(images_timisoara.keys()))

# --- KITTI-C (cadre reale + ploaie sintetică pe imaginea curată) ---
kitti_clean_np = np.array(Image.open('samples/kitti_c_clean.png').convert('RGB'))

images_kitti = {
    'Original':           kitti_clean_np,
    'Ceață (sev. 5)':     np.array(Image.open('samples/kitti_c_fog5.png').convert('RGB')),
    'Ninsoare (sev. 3)':  np.array(Image.open('samples/kitti_c_snow3.png').convert('RGB')),
    'Chiciură (sev. 3)':  np.array(Image.open('samples/kitti_c_frost3.png').convert('RGB')),
    'Ploaie (sint.)':     np.array(apply_rain(Image.fromarray(kitti_clean_np))),
}
print('KITTI-C pregătit:', list(images_kitti.keys()))

In [ ]:
# 7. Funcții auxiliare de inferență și vizualizare
def run_inference(encoder, decoder, pil_img):
    img_np = np.array(pil_img.resize((WIDTH, HEIGHT))).astype(np.float32) / 255.0
    inp = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        output = decoder(encoder(inp), raw_image=inp)
    disp, _ = disp_to_depth(output[('disp', 0)][:, 0:1], 0.1, 100.0)
    disp_np = disp[0, 0].cpu().numpy()
    sigma = (
        torch.sigmoid(output[('uncert', 0)])[0, 0].cpu().numpy()
        if ('uncert', 0) in output
        else np.zeros_like(disp_np)
    )
    return disp_np, sigma

def run_inference_np(encoder, decoder, img_np_array):
    return run_inference(encoder, decoder, Image.fromarray(img_np_array))

def make_heatmap(d):
    d_norm = (d - d.min()) / (d.max() - d.min() + 1e-8)
    bgr = cv2.applyColorMap((d_norm * 255).astype(np.uint8), cv2.COLORMAP_MAGMA)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

print('Funcții auxiliare definite.')

In [ ]:
# 8. Inferență Figura 1 — Timișoara cu vreme sintetică
weather_names = list(images_timisoara.keys())
results_tim = {}
for name in weather_names:
    d_u, s_u = run_inference(enc_urw, dec_urw, images_timisoara[name])
    d_w, s_w = run_inference(enc_w,   dec_w,   images_timisoara[name])
    results_tim[name] = {
        'urw':     {'disp': d_u, 'sigma': s_u},
        'weather': {'disp': d_w, 'sigma': s_w},
    }
    print(f'  {name}: OK')

# Vizualizare: 5 rânduri × 4 coloane
row_labels = [
    'Imagine intrare',
    'URW-Depth\nDisparitate',
    'URW-Depth\nIncertitudine (σ)',
    'URW-Depth-Weather\nDisparitate',
    'URW-Depth-Weather\nIncertitudine (σ)',
]

fig, axes = plt.subplots(5, 4, figsize=(5.5 * 4, 14))
fig.suptitle(
    'URW-Depth vs URW-Depth-Weather — Disparitate și Incertitudine în condiții adverse',
    fontsize=14, fontweight='bold', y=1.01
)

for col, name in enumerate(weather_names):
    r = results_tim[name]
    axes[0, col].imshow(images_timisoara[name].resize((WIDTH, HEIGHT)))
    axes[0, col].set_title(name, fontsize=13, fontweight='bold')
    axes[0, col].axis('off')

    axes[1, col].imshow(make_heatmap(r['urw']['disp']))
    axes[1, col].axis('off')

    im_s1 = axes[2, col].imshow(r['urw']['sigma'], cmap='inferno', vmin=0, vmax=1)
    axes[2, col].axis('off')
    plt.colorbar(im_s1, ax=axes[2, col], fraction=0.046, pad=0.04)

    axes[3, col].imshow(make_heatmap(r['weather']['disp']))
    axes[3, col].axis('off')

    im_s2 = axes[4, col].imshow(r['weather']['sigma'], cmap='inferno', vmin=0, vmax=1)
    axes[4, col].axis('off')
    plt.colorbar(im_s2, ax=axes[4, col], fraction=0.046, pad=0.04)

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=11, rotation=0, labelpad=140, va='center')

plt.tight_layout()
plt.savefig('figura1_timisoara.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvat: figura1_timisoara.png')

In [ ]:
# 9. Inferență + Figura 2 — Comparație robusteță KITTI-C
kitti_names = list(images_kitti.keys())
results_kitti = {}
for name, img_np in images_kitti.items():
    d_t, _ = run_inference_np(enc_tiny, dec_tiny, img_np)
    d_u, _ = run_inference_np(enc_urw,  dec_urw,  img_np)
    d_w, _ = run_inference_np(enc_w,    dec_w,    img_np)
    results_kitti[name] = {'tiny': d_t, 'urw': d_u, 'weather': d_w}
    print(f'  {name}: OK')

# Vizualizare: 4 rânduri × 5 coloane
n_cols = len(kitti_names)
row_labels_k = [
    'Imagine\nintrare',
    'Tiny-Depth\n(baseline)',
    'URW-Depth\n(calibrat)',
    'URW-Depth-Weather\n(robust)',
]

fig = plt.figure(figsize=(4.5 * n_cols, 3.2 * 4))
left_m = 0.13
gs = fig.add_gridspec(4, n_cols, left=left_m, right=0.99,
                       top=0.96, bottom=0.02, hspace=0.05, wspace=0.03)
axes = gs.subplots()

for ci, name in enumerate(kitti_names):
    r = results_kitti[name]
    img_show = cv2.resize(images_kitti[name], (WIDTH, HEIGHT))
    axes[0, ci].imshow(img_show)
    axes[0, ci].set_title(name, fontsize=11, fontweight='bold', pad=5)
    axes[0, ci].axis('off')
    axes[1, ci].imshow(make_heatmap(r['tiny']));    axes[1, ci].axis('off')
    axes[2, ci].imshow(make_heatmap(r['urw']));     axes[2, ci].axis('off')
    axes[3, ci].imshow(make_heatmap(r['weather'])); axes[3, ci].axis('off')

for ri, lab in enumerate(row_labels_k):
    pos = axes[ri, 0].get_position()
    fig.text(
        left_m - 0.01,
        pos.y0 + pos.height / 2,
        lab, fontsize=11, ha='right', va='center', linespacing=1.5
    )

plt.savefig('figura2_kitti_robustete.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvat: figura2_kitti_robustete.png')

In [ ]:
# 10. (Opțional) Descarcă figurile
from google.colab import files as colab_files
colab_files.download('figura1_timisoara.png')
colab_files.download('figura2_kitti_robustete.png')